# Visualization Examples

This notebook demonstrates the cleaned visualization modules for Savvy. It intentionally uses placeholder paths and guarded execution flags so the notebook can be committed without local dataset assumptions.

Covered examples:
- RGB / GT / prediction sequence strips
- prediction-reference support matrix
- OGA behavior matrix
- GT reassociation curves from `identity_events.json`


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Make imports work whether the notebook is launched from repo root or notebooks/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "visualization").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from visualization.sequence_strips import (
    find_frame_dir,
    infer_scannet_sampling_rule,
    visualize_gt_strip_matched_to_prediction,
    visualize_prediction_strip,
    visualize_rgb_strip_matched_to_prediction,
)
from visualization.support_matrix_behavior_matrix import (
    compute_prediction_reference_support_matrix,
    load_scannet_scene_for_support_matrix,
    plot_behavior_matrix,
    plot_support_degrees,
    plot_support_matrix,
    summarize_support_matrix,
)
from visualization.identity_discovery_reassociation_events import (
    plot_multimethod_reassociation_curve,
)

REPO_ROOT


## Placeholder Paths

Replace these with local dataset, prediction, and analysis directories before running the examples.

In [ ]:
SCENE_ID = "scene0000_00"

# ScanNet-style folders.
SCANNET_GT_ROOT = Path("/path/to/scannet/gt")
SCANNET_RGB_ROOT = Path("/path/to/scannet/val")
PRED_ROOT = Path("/path/to/eval_predictions/Savvy")

# Scene-specific folders used by sequence-strip examples.
GT_INSTANCE_DIR = SCANNET_GT_ROOT / SCENE_ID / "instance"
PRED_SCENE_DIR = PRED_ROOT / SCENE_ID

# Evaluation CSVs used by the behavior matrix.
METHOD_CSVS = {
    "Savvy": "/path/to/savvy/oga_full_results.csv",
    "DEVA+SAM": "/path/to/deva/oga_full_results.csv",
    "EntitySAM": "/path/to/entitysam/oga_full_results.csv",
}

# Folders containing identity_events.json for each method.
REASSOC_METHODS = {
    "Savvy": {
        "analysis_dir": "/path/to/savvy_identity_event_analysis",
        "color_reappear": "#74E3D8",
        "color_success": "#6577D9",
    },
    "DEVA+SAM": {
        "analysis_dir": "/path/to/deva_identity_event_analysis",
        "color_reappear": "#D28B5A",
        "color_success": "#4F9D69",
    },
    "EntitySAM": {
        "analysis_dir": "/path/to/entitysam_identity_event_analysis",
        "color_reappear": "#B07CC6",
        "color_success": "#4B8AC9",
    },
}


## Sequence Strips

This example creates aligned prediction, GT, and RGB strips. Set `RUN_SEQUENCE_STRIPS = True` after replacing the placeholder paths.

In [ ]:
RUN_SEQUENCE_STRIPS = False

if RUN_SEQUENCE_STRIPS:
    original_interval, cutoff, num_gt_frames = infer_scannet_sampling_rule(GT_INSTANCE_DIR)
    print(f"GT frames: {num_gt_frames}, cutoff: {cutoff}, original interval: {original_interval}")

    pred_strip, pred_lut, pred_pngs, pred_indices, fig, ax = visualize_prediction_strip(
        PRED_SCENE_DIR,
        num_frames=8,
        seed=0,
        layout="horizontal",
        show_frame_names=True,
        save_path=None,
    )

    gt_strip, gt_lut, gt_pngs, gt_indices, fig, ax = visualize_gt_strip_matched_to_prediction(
        GT_INSTANCE_DIR,
        pred_indices,
        original_interval=original_interval,
        seed=0,
        layout="horizontal",
        show_frame_names=True,
        save_path=None,
    )

    frame_dir = find_frame_dir(SCANNET_RGB_ROOT, SCENE_ID)
    rgb_strip, rgb_paths, rgb_indices, fig, ax = visualize_rgb_strip_matched_to_prediction(
        frame_dir,
        pred_indices,
        original_interval=original_interval,
        layout="horizontal",
        show_frame_names=False,
        save_path=None,
    )


## Support Matrix

This example loads one ScanNet scene, computes prediction-reference support, and plots GT-normalized and prediction-normalized matrices.

In [ ]:
RUN_SUPPORT_MATRIX = False

if RUN_SUPPORT_MATRIX:
    preds, gts, frame_indices = load_scannet_scene_for_support_matrix(
        scene_id=SCENE_ID,
        gt_dir=SCANNET_GT_ROOT,
        pred_root=PRED_ROOT,
    )

    support = compute_prediction_reference_support_matrix(
        preds,
        gts,
        ignore_label=0,
        iou_thr=0.5,
        ios_thr=0.5,
        use_void_tolerant=True,
    )

    summarize_support_matrix(support)


In [ ]:
if RUN_SUPPORT_MATRIX:
    fig, ax, sliced = plot_support_matrix(
        support,
        normalize_by="gt",
        orientation="ref_pred",
        sort_mode="dominant_gt",
        top_rows=80,
        top_cols=80,
        cmap="support_default",
        title=f"{SCENE_ID}: GT-normalized support",
        save_path=None,
    )

    fig, ax, sliced = plot_support_matrix(
        support,
        normalize_by="pred",
        orientation="ref_pred",
        sort_mode="dominant_gt",
        top_rows=80,
        top_cols=80,
        cmap="support_default",
        title=f"{SCENE_ID}: prediction-normalized support",
        save_path=None,
    )

    fig, axes, sliced = plot_support_degrees(
        support,
        top_rows=80,
        top_cols=80,
        save_path=None,
    )


## Behavior Matrix

This plot compares methods in the `IC (P)` / `IC (G)` plane using evaluation CSVs.

In [ ]:
RUN_BEHAVIOR_MATRIX = False

if RUN_BEHAVIOR_MATRIX:
    fig, ax = plot_behavior_matrix(
        METHOD_CSVS,
        threshold=0.2,
        title="OGA Tracking Behavior Topology Matrix",
        save_path=None,
    )


## Reassociation Events

This example plots GT reappearance and successful reassociation curves from saved `identity_events.json` folders. Discovery curves are intentionally omitted here.

In [ ]:
RUN_REASSOCIATION_CURVES = False

if RUN_REASSOCIATION_CURVES:
    fig, ax = plot_multimethod_reassociation_curve(
        REASSOC_METHODS,
        num_grid_points=1000,
        plot_threads=True,
        title="ScanNet: GT Reappearance and Successful Reassociation",
        save_path=None,
    )
